# Proyek Sains Data - Ekstraksi Fitur Citra

Notebook ini melakukan ekstraksi fitur warna (RGB) dari dataset citra pada folder `Citra`:
- **Fitur Warna (RGB)**:
  - Rata-rata (Mean): `mean_r`, `mean_g`, `mean_b`
  - Standar Deviasi (Standard Deviation): `std_r`, `std_g`, `std_b`
  - Kemiringan Distribusi (Skewness): `skew_r`, `skew_g`, `skew_b`
- **Label / Target**:
  - `kelas`: Diambil otomatis dari nama subfolder (`rapi`, `berantakan`)

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from scipy.stats import skew

# 1. Path folder dataset citra (mendukung 'Citra' atau 'citra')
folder_citra = 'Citra' if os.path.exists('Citra') else 'citra'

# Format file citra yang didukung
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

# List untuk menampung hasil ekstraksi fitur
data_fitur = []

# 2. Iterasi setiap sub-folder (kelas) dan file gambar di dalam folder citra
for root, dirs, files in os.walk(folder_citra):
    for file in files:
        if file.lower().endswith(valid_extensions):
            file_path = os.path.join(root, file)
            
            # Kelas didapat dari nama folder tempat citra berada (misal: 'rapi', 'berantakan')
            kelas = os.path.basename(root)
            
            # Membaca citra menggunakan OpenCV (format bawaan: BGR)
            img_bgr = cv2.imread(file_path)
            
            # Fallback jika cv2.imread mengembalikan None (misal handling path tertentu)
            if img_bgr is None:
                img_bgr = cv2.imdecode(np.fromfile(file_path, dtype=np.uint8), cv2.IMREAD_COLOR)
                
            if img_bgr is not None:
                # Mengonversi format BGR ke RGB
                img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
                
                # Menghitung rata-rata (Mean) untuk setiap channel (R, G, B)
                # axis=(0, 1) menghitung rata-rata pada dimensi tinggi (height) dan lebar (width)
                mean_r, mean_g, mean_b = img_rgb.mean(axis=(0, 1))
                
                # Menghitung standar deviasi (Std) untuk setiap channel (R, G, B)
                std_r, std_g, std_b = img_rgb.std(axis=(0, 1))
                
                # Menghitung kemiringan (Skewness) untuk setiap channel (R, G, B)
                skew_r, skew_g, skew_b = skew(img_rgb.reshape(-1, 3), axis=0)
                
                # Menyimpan data fitur ke list
                data_fitur.append({
                    'nama_file': file,
                    'mean_r': mean_r,
                    'mean_g': mean_g,
                    'mean_b': mean_b,
                    'std_r': std_r,
                    'std_g': std_g,
                    'std_b': std_b,
                    'skew_r': skew_r,
                    'skew_g': skew_g,
                    'skew_b': skew_b,
                    'kelas': kelas
                })

# 3. Membuat DataFrame dari hasil ekstraksi
df = pd.DataFrame(data_fitur)

print(f"Total citra yang berhasil diekstraksi: {len(df)}")
df.head()

In [2]:
# Menampilkan informasi ringkas dataset dan jumlah sampel per kelas
print("Informasi Dataset:")
df.info()

print("\nJumlah Sampel per Kelas:")
print(df['kelas'].value_counts())

Informasi Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192 entries, 0 to 191
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   nama_file  192 non-null    object 
 1   mean_r     192 non-null    float64
 2   mean_g     192 non-null    float64
 3   mean_b     192 non-null    float64
 4   std_r      192 non-null    float64
 5   std_g      192 non-null    float64
 6   std_b      192 non-null    float64
 7   skew_r     192 non-null    float64
 8   skew_g     192 non-null    float64
 9   skew_b     192 non-null    float64
 10  kelas      192 non-null    object 
dtypes: float64(9), object(2)
memory usage: 16.6+ KB

Jumlah Sampel per Kelas:
kelas
berantakan    96
rapi          96
Name: count, dtype: int64


In [3]:
# Menyimpan hasil ekstraksi fitur ke file CSV
output_csv = 'hasil_ekstraksi_fitur.csv'
try:
    df.to_csv(output_csv, index=False)
    print(f"Hasil ekstraksi fitur berhasil disimpan ke '{output_csv}'")
except PermissionError:
    # Jika file utama sedang dibuka di aplikasi lain (seperti Microsoft Excel)
    alt_csv = 'hasil_ekstraksi_fitur_terbaru.csv'
    df.to_csv(alt_csv, index=False)
    print(f"Perhatian: '{output_csv}' sedang dibuka oleh aplikasi lain (misal: Microsoft Excel).")
    print(f"Hasil ekstraksi berhasil disimpan ke '{alt_csv}'. Tutup Excel jika ingin menimpa file utama.")

Perhatian: 'hasil_ekstraksi_fitur.csv' sedang dibuka oleh aplikasi lain (misal: Microsoft Excel).
Hasil ekstraksi berhasil disimpan ke 'hasil_ekstraksi_fitur_terbaru.csv'. Tutup Excel jika ingin menimpa file utama.
